# Expression Profiles

## PI3K-AKT

In [30]:
library(tidyverse)

generate_expression_plots <- function(input_file, output_dir, 
                                      condition_patterns, condition_colors, 
                                      gene_categories) {
  # Step 1: Load data
  data <- read_csv(input_file)
  
  # Step 2: Pivot to long format and assign Condition dynamically and safely
  long_data <- data %>%
    pivot_longer(
      cols = -GENE_ID,
      names_to = "Sample",
      values_to = "Expression"
    ) %>%
    mutate(Condition = {
      cond_vec <- rep("Other", length(Sample))
      for (cond_name in names(condition_patterns)) {
        pattern <- condition_patterns[[cond_name]]
        if (!is.null(pattern) && is.character(pattern)) {
          idx <- str_detect(Sample, pattern) & cond_vec == "Other"
          cond_vec[idx] <- cond_name
        }
      }
      cond_vec
    }) %>%
    mutate(Condition = factor(Condition, levels = names(condition_colors)))

  
  # Create output directory if missing
  if(!dir.exists(output_dir)) dir.create(output_dir, recursive = TRUE)
  
  # Step 3: For each gene category, generate and save plot
  for (category_name in names(gene_categories)) {
    genes <- gene_categories[[category_name]]
    
    plot_data <- long_data %>%
      filter(GENE_ID %in% genes)
    
    if (nrow(plot_data) == 0) next
    
    p <- ggplot(plot_data, aes(x = GENE_ID, y = Expression, color = Condition)) +
      geom_jitter(width = 0.2, alpha = 0.7, size = 2) +
      stat_summary(fun = mean, geom = "crossbar", width = 0.5, fatten = 2, color = "black") +
      scale_color_manual(values = condition_colors) +
      labs(
        title = paste0(basename(input_file), " - ", gsub("_", " ", category_name)),
        y = "Expression Level",
        x = "Gene"
      ) +
      theme_minimal(base_size = 14) +
      theme(
        axis.text.x = element_text(angle = 45, hjust = 1, size = 12),
        axis.title.x = element_text(size = 14),
        plot.title = element_text(hjust = 0.5),
        legend.position = "bottom"
      )
    
    output_path <- file.path(output_dir, paste0(category_name, "_expression_plot.png"))
    ggsave(output_path, plot = p, width = 10, height = 6, dpi = 300)
  }
}

gene_categories <- gene_lists <- list(
  
    "Common genes" = c(
    "GAB1", "SMAD2", "CCND2", "PPP1R12B", "KCNJ13"
  ),
  
  "PI3K-AKT singalling" = c(
    "PIK3CA", "PIK3R1", "AKT", "PTEN", "mTOR", "VEGF", "EGFR", "MYC", "KIT", "ERBB2", "MET", "FGFR2", "FGFR3", "PDGFRA"
  ),
  
  "Integrins" = c(
    "ITGA1", "ITGA2", "ITGA3", "ITGA4", "ITGA5", "ITGA6", "ITGA7", "ITGA8", "ITGA9", "ITGA10", "ITGA11",
  "ITGAD", "ITGAE", "ITGAL", "ITGAM", "ITGAV", "ITGA2B",
  "ITGB1", "ITGB2", "ITGB3", "ITGB4", "ITGB5", "ITGB6", "ITGB7", "ITGB8"
  )
)

In [31]:
generate_expression_plots(
  input_file = "Normalized/GSE38265.csv",
  output_dir = "DEG/Expression profiles/PI3K-AKT/GSE38265",
  condition_patterns = list(terminal = "terminal", ESC = "ESC", iPSC = "iPSC"),
  condition_colors = c(iPSC = "red", ESC = "green4", terminal = "blue"),
  gene_categories = gene_categories
)

Rows: 22340 Columns: 7
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (1): GENE_ID
dbl (6): BJ_terminal_GSE38265, iM2_iPSC_GSE38265, iM21_iPSC_GSE38265, iG2_iP...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [32]:
generate_expression_plots(
  input_file = "Normalized/GSE69626.csv",
  output_dir = "DEG/Expression profiles/PI3K-AKT/GSE69626",
  condition_patterns = list("hiPSC" = "^.*hiPSC.*$", "iPSC" = "^.*iPSC.*$", "ESC" = "ESC", "terminal" = "terminal"),
  condition_colors = c("iPSC" = "red", "ESC" = "green4", "hiPSC" = "blue"),
  gene_categories = gene_categories
)

Rows: 39376 Columns: 28
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr  (1): GENE_ID
dbl (27): GSM1706658_hiPSC_GSE69626, GSM1706663_hiPSC_GSE69626, GSM1706668_h...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [33]:
generate_expression_plots(
  input_file = "Normalized/GSE239446.csv",
  output_dir = "DEG/Expression profiles/PI3K-AKT/GSE239446",
  condition_patterns = list(NSC = "NSC", Reprogrammed = "Reprogrammed", iPSC = "iPSC"),
  condition_colors = c("iPSC" = "red", "Reprogrammed" = "green4", "NSC" = "blue"),
  gene_categories = gene_categories
)

Rows: 62707 Columns: 13
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr  (1): GENE_ID
dbl (12): GSM8422193_iPSC_GSE239446, GSM8422194_iPSC_GSE239446, GSM8422195_i...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [36]:
generate_expression_plots(
  input_file = "Normalized/GSE251814.csv",
  output_dir = "DEG/Expression profiles/PI3K-AKT/GSE251814",
  condition_patterns = list("hiPSC" = "^.*hiPSC.*$", "iPSC" = "^.*iPSC.*$"),
  condition_colors = c("hiPSC" = "red", "iPSC" = "blue", Other = "grey"),
  gene_categories = gene_categories
)

Rows: 60660 Columns: 17
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr  (1): GENE_ID
dbl (16): AG25367_g2_CTL_d4_hiPSC_GSE251814, AG25367_g4_CTL_d4_hiPSC_GSE2518...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


## Mitochondria

In [38]:
library(tidyverse)

generate_expression_plots <- function(input_file, output_dir, condition_colors, condition_patterns) {
  data <- read_csv(input_file)
  
  long_data <- data %>%
    pivot_longer(
      cols = -GENE_ID,
      names_to = "Sample",
      values_to = "Expression"
    ) %>%
    mutate(Condition = {
      cond_vec <- rep("Other", length(Sample))
      for (cond_name in names(condition_patterns)) {
        pattern <- condition_patterns[[cond_name]]
        if (!is.null(pattern) && is.character(pattern)) {
          idx <- str_detect(Sample, pattern) & cond_vec == "Other"
          cond_vec[idx] <- cond_name
        }
      }
      cond_vec
    })
  
  gene_categories <- list(
    "Tracing Mitochondrial Markers" = c(
      "TFAM", "VDAC1", "COX1", "COX2", "MT-CO1", "MT-ATP6",
      "MT-ND1", "MT-ND2", "MT-ND3", "MT-ND4", "MT-ND5", "MT-ND6",
      "MT-CYB", "SDHA", "SDHB", "SDHC", "SDHD",
      "NDUFB8", "UQCRC2", "ATP5A"
    ),
    "Alternative Splicing" = c("ESRP1"),
    "Mitochondrial Dysfunction and Global Variability" = c(
      "NRF2", "NFE2L2", "PARP1", "GLDC",
      "POLG", "POLG2", "TWNK", "C10orf2",
      "TK2", "DGUOK", "SUCLA2", "SUCLG1", "RRM2B", "TYMP",
      "MPV17", "OPA1", "MFN1", "MFN2", "DRP1", "DNM1L", "FIS1"
    ),
    "Bioenergetics from Gene Analysis" = c(
      "HK2", "PKM1", "PKM2", "MYC", "HIF1A",
      "NDUFS1", "NDUFS2", "NDUFS3", "NDUFS4", "NDUFS5", "NDUFS6", "NDUFS7", "NDUFS8",
      "NDUFV1", "NDUFV2", "BCS1L",
      "SLC25A4", "SLC25A5", "UCP1", "UCP2", "PGC1A"
    ),
    "Mitochondrial Rejuvenation or Exhaustion" = c(
      "PINK1", "PARK2", "NIX", "ULK1", "AMBRA1",
      "SIRT1", "SIRT3", "SOD2", "CAT", "GPX1",
      "MT-TL1", "MT-TK", "MT-TI", "MT-TW", "MT-TG"
    ),
    "Integrins and Sestrins" = c(
      "ITGA1", "ITGA2", "ITGA3", "ITGA4", "ITGA5", "ITGA6", "ITGA7", "ITGA8", "ITGA9", "ITGA10", "ITGA11",
      "ITGAD", "ITGAE", "ITGAL", "ITGAM", "ITGAV", "ITGA2B",
      "ITGB1", "ITGB2", "ITGB3", "ITGB4", "ITGB5", "ITGB6", "ITGB7", "ITGB8"
    )
  )
  
  cat("Condition assignments in data:\n")
  print(table(long_data$Condition))
  
  for (category_name in names(gene_categories)) {
    genes <- gene_categories[[category_name]]
    plot_data <- long_data %>%
      filter(GENE_ID %in% genes) %>%
      mutate(Expression_log2 = log2(Expression + 1))
    
    if (nrow(plot_data) == 0) next
    
    cat("Processing:", category_name, "- Found", n_distinct(plot_data$GENE_ID), "genes\n")
    
    p <- ggplot(plot_data, aes(x = GENE_ID, y = Expression_log2, color = Condition)) +
      geom_jitter(width = 0.2, alpha = 0.7, size = 2) +
      stat_summary(fun = mean, geom = "crossbar", width = 0.5, fatten = 2, color = "black") +
      scale_color_manual(values = condition_colors) +
      labs(
        title = paste0(basename(input_file), " - ", gsub("_", " ", category_name)),
        y = "Expression Level (log2)",
        x = "Gene"
      ) +
      theme_minimal(base_size = 14) +
      theme(
        axis.text.x = element_text(angle = 45, hjust = 1, size = 12),
        axis.title.x = element_text(size = 14),
        plot.title = element_text(hjust = 0.5),
        legend.position = "bottom"
      )
    
    if (!dir.exists(output_dir)) dir.create(output_dir, recursive = TRUE)
    output_path <- file.path(output_dir, paste0(category_name, "_expression_plot.png"))
    ggsave(output_path, plot = p, width = 10, height = 6, dpi = 300)
    cat("Saved:", output_path, "\n")
  }
}

In [39]:
# For GSE38265
generate_expression_plots(
  input_file = "Normalized/GSE38265.csv",
  output_dir = "DEG/Expression profiles/Mitochondria/GSE38265",
  condition_colors = c("iPSC" = "red", "ESC" = "green4", "terminal" = "blue"),
  condition_patterns = list("terminal" = "terminal", "ESC" = "ESC", "iPSC" = "iPSC")
)

Rows: 22340 Columns: 7
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (1): GENE_ID
dbl (6): BJ_terminal_GSE38265, iM2_iPSC_GSE38265, iM21_iPSC_GSE38265, iG2_iP...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


Condition assignments in data:

     ESC     iPSC terminal 
   22340    89360    22340 
Processing: Tracing Mitochondrial Markers - Found 8 genes
Saved: DEG/Expression profiles/Mitochondria/GSE38265/Tracing Mitochondrial Markers_expression_plot.png 
Processing: Alternative Splicing - Found 1 genes
Saved: DEG/Expression profiles/Mitochondria/GSE38265/Alternative Splicing_expression_plot.png 
Processing: Mitochondrial Dysfunction and Global Variability - Found 18 genes
Saved: DEG/Expression profiles/Mitochondria/GSE38265/Mitochondrial Dysfunction and Global Variability_expression_plot.png 
Processing: Bioenergetics from Gene Analysis - Found 19 genes
Saved: DEG/Expression profiles/Mitochondria/GSE38265/Bioenergetics from Gene Analysis_expression_plot.png 
Processing: Mitochondrial Rejuvenation or Exhaustion - Found 9 genes
Saved: DEG/Expression profiles/Mitochondria/GSE38265/Mitochondrial Rejuvenation or Exhaustion_expression_plot.png 
Processing: Integrins and Sestrins - Found 25 genes


In [40]:
# For GSE69626
generate_expression_plots(
  input_file = "Normalized/GSE69626.csv",
  output_dir = "DEG/Expression profiles/Mitochondria/GSE69626",
  condition_colors = c("iPSC" = "red", "ESC" = "green4", "hiPSC" = "blue"),
  condition_patterns = list("hiPSC" = "^.*hiPSC.*$", "iPSC" = "^.*iPSC.*$", "ESC" = "ESC")
)

Rows: 39376 Columns: 28
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr  (1): GENE_ID
dbl (27): GSM1706658_hiPSC_GSE69626, GSM1706663_hiPSC_GSE69626, GSM1706668_h...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


Condition assignments in data:

   ESC  hiPSC   iPSC  Other 
118128 236256 669392  39376 
Processing: Tracing Mitochondrial Markers - Found 17 genes
Saved: DEG/Expression profiles/Mitochondria/GSE69626/Tracing Mitochondrial Markers_expression_plot.png 
Processing: Alternative Splicing - Found 1 genes
Saved: DEG/Expression profiles/Mitochondria/GSE69626/Alternative Splicing_expression_plot.png 
Processing: Mitochondrial Dysfunction and Global Variability - Found 18 genes
Saved: DEG/Expression profiles/Mitochondria/GSE69626/Mitochondrial Dysfunction and Global Variability_expression_plot.png 
Processing: Bioenergetics from Gene Analysis - Found 18 genes
Saved: DEG/Expression profiles/Mitochondria/GSE69626/Bioenergetics from Gene Analysis_expression_plot.png 
Processing: Mitochondrial Rejuvenation or Exhaustion - Found 8 genes
Saved: DEG/Expression profiles/Mitochondria/GSE69626/Mitochondrial Rejuvenation or Exhaustion_expression_plot.png 
Processing: Integrins and Sestrins - Found 25 gen

In [41]:
# For GSE251814
generate_expression_plots(
  input_file = "Normalized/GSE251814.csv",
  output_dir = "DEG/Expression profiles/Mitochondria/GSE251814",
  condition_colors = c("hiPSC" = "red", "iPSC" = "blue"),
  condition_patterns = list("hiPSC" = "^.*hiPSC.*$", "iPSC" = "^.*iPSC.*$")
)

Rows: 60660 Columns: 17
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr  (1): GENE_ID
dbl (16): AG25367_g2_CTL_d4_hiPSC_GSE251814, AG25367_g4_CTL_d4_hiPSC_GSE2518...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


Condition assignments in data:

 hiPSC   iPSC 
485280 485280 
Processing: Tracing Mitochondrial Markers - Found 17 genes
Saved: DEG/Expression profiles/Mitochondria/GSE251814/Tracing Mitochondrial Markers_expression_plot.png 
Processing: Alternative Splicing - Found 1 genes
Saved: DEG/Expression profiles/Mitochondria/GSE251814/Alternative Splicing_expression_plot.png 
Processing: Mitochondrial Dysfunction and Global Variability - Found 18 genes
Saved: DEG/Expression profiles/Mitochondria/GSE251814/Mitochondrial Dysfunction and Global Variability_expression_plot.png 
Processing: Bioenergetics from Gene Analysis - Found 18 genes
Saved: DEG/Expression profiles/Mitochondria/GSE251814/Bioenergetics from Gene Analysis_expression_plot.png 
Processing: Mitochondrial Rejuvenation or Exhaustion - Found 13 genes
Saved: DEG/Expression profiles/Mitochondria/GSE251814/Mitochondrial Rejuvenation or Exhaustion_expression_plot.png 
Processing: Integrins and Sestrins - Found 25 genes
Saved: DEG/Expressi

In [ ]:
generate_expression_plots(
  input_file = "Normalized/GSE104406.csv",
  output_dir = "DEG/Expression profiles/Mitochondria/GSE104406",
  condition_colors = c("young" = "red", "old" = "blue"),
  condition_patterns = list("young" = "young", "old" = "old")
)

Rows: 57260 Columns: 21
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr  (1): GENE_ID
dbl (20): HSC_GSE104406_young1_GSM2797287_Sample_56677, HSC_GSE104406_young1...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


Condition assignments in data:

   old  young 
572600 572600 
Processing: Tracing Mitochondrial Markers - Found 8 genes
Saved: DEG/Expression profiles/Mitochondria/GSE104406/Tracing Mitochondrial Markers_expression_plot.png 
Processing: Alternative Splicing - Found 1 genes
Saved: DEG/Expression profiles/Mitochondria/GSE104406/Alternative Splicing_expression_plot.png 
Processing: Mitochondrial Dysfunction and Global Variability - Found 18 genes
Saved: DEG/Expression profiles/Mitochondria/GSE104406/Mitochondrial Dysfunction and Global Variability_expression_plot.png 
Processing: Bioenergetics from Gene Analysis - Found 18 genes
Saved: DEG/Expression profiles/Mitochondria/GSE104406/Bioenergetics from Gene Analysis_expression_plot.png 
Processing: Mitochondrial Rejuvenation or Exhaustion - Found 7 genes
Saved: DEG/Expression profiles/Mitochondria/GSE104406/Mitochondrial Rejuvenation or Exhaustion_expression_plot.png 
Processing: Integrins and Sestrins - Found 25 genes
Saved: DEG/Expression

In [ ]:
generate_expression_plots(
  input_file = "Normalized/GSE113253.csv",
  output_dir = "DEG/Expression profiles/Mitochondria/GSE113253",
  condition_colors = c("AT" = "red", "BM" = "green4", "Msc" = "blue"),
  condition_patterns = list("AT" = "AT", "BM" = "BM", "Msc" = "Msc")
)

Rows: 19946 Columns: 153
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr   (1): GENE_ID
dbl (152): RNA_14dOb_AT_rep1, RNA_7dOb_AT_rep1, RNA_3dOb_AT_rep1, RNA_1dOb_A...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


Condition assignments in data:

     AT      BM     Msc   Other 
1077084 1276544  398920  279244 


Warning message:
"There was 1 warning in `mutate()`.
ℹ In argument: `Expression_log2 = log2(Expression + 1)`.
Caused by warning:
! NaNs produced"


Processing: Tracing Mitochondrial Markers - Found 8 genes


Warning message:
"Removed 10 rows containing non-finite outside the scale range
(`stat_summary()`)."
Warning message:
"Removed 10 rows containing missing values or values outside the scale range
(`geom_point()`)."


...

Saved: DEG/Expression profiles/Mitochondria/GSE113253/Mitochondrial Rejuvenation or Exhaustion_expression_plot.png 


Warning message:
"There was 1 warning in `mutate()`.
ℹ In argument: `Expression_log2 = log2(Expression + 1)`.
Caused by warning:
! NaNs produced"


Processing: Integrins and Sestrins - Found 24 genes


Warning message:
"Removed 282 rows containing non-finite outside the scale range
(`stat_summary()`)."
Warning message:
"Removed 282 rows containing missing values or values outside the scale range
(`geom_point()`)."


Saved: DEG/Expression profiles/Mitochondria/GSE113253/Integrins and Sestrins_expression_plot.png 


In [9]:
generate_expression_plots(
  input_file = "Normalized/GSE139273.csv",
  output_dir = "DEG/Expression profiles/Mitochondria/GSE139273",
  condition_colors = c("iPSC" = "red", "hDRG" = "green4", "iSN" = "blue"),
  condition_patterns = list("iPSC" = "iPSC", "hDRG" = "hDRG", "iSN" = "iSN")
)

Rows: 60448 Columns: 8
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (1): GENE_ID
dbl (7): neuron_GSE139273_control_hDRG_1_GSM4135900, iPSC_GSE139273_iPSC_1_G...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


Condition assignments in data:

  hDRG   iPSC    iSN 
 60448 181344 181344 
Processing: Tracing Mitochondrial Markers - Found 17 genes
Saved: DEG/Expression profiles/Mitochondria/GSE139273/Tracing Mitochondrial Markers_expression_plot.png 
Processing: Alternative Splicing - Found 1 genes
Saved: DEG/Expression profiles/Mitochondria/GSE139273/Alternative Splicing_expression_plot.png 
Processing: Mitochondrial Dysfunction and Global Variability - Found 18 genes
Saved: DEG/Expression profiles/Mitochondria/GSE139273/Mitochondrial Dysfunction and Global Variability_expression_plot.png 
Processing: Bioenergetics from Gene Analysis - Found 18 genes
Saved: DEG/Expression profiles/Mitochondria/GSE139273/Bioenergetics from Gene Analysis_expression_plot.png 
Processing: Mitochondrial Rejuvenation or Exhaustion - Found 14 genes
Saved: DEG/Expression profiles/Mitochondria/GSE139273/Mitochondrial Rejuvenation or Exhaustion_expression_plot.png 
Processing: Integrins and Sestrins - Found 25 genes
Saved

In [10]:
generate_expression_plots(
  input_file = "Normalized/GSE164425.csv",
  output_dir = "DEG/Expression profiles/Mitochondria/GSE164425",
  condition_colors = c("NK" = "red", "HSCinduced" = "blue"),
  condition_patterns = list("NK" = "NK", "HSCinduced" = "HSCinduced")
)

Rows: 38692 Columns: 23
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr  (1): GENE_ID
dbl (22): Tcell_GSE164425_Activate_gamadeltaT_4, Tcell_GSE164425_Activate_ga...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


Condition assignments in data:

HSCinduced         NK      Other 
    232152      77384     541688 
Processing: Tracing Mitochondrial Markers - Found 17 genes
Saved: DEG/Expression profiles/Mitochondria/GSE164425/Tracing Mitochondrial Markers_expression_plot.png 
Processing: Alternative Splicing - Found 1 genes
Saved: DEG/Expression profiles/Mitochondria/GSE164425/Alternative Splicing_expression_plot.png 
Processing: Mitochondrial Dysfunction and Global Variability - Found 18 genes
Saved: DEG/Expression profiles/Mitochondria/GSE164425/Mitochondrial Dysfunction and Global Variability_expression_plot.png 
Processing: Bioenergetics from Gene Analysis - Found 17 genes
Saved: DEG/Expression profiles/Mitochondria/GSE164425/Bioenergetics from Gene Analysis_expression_plot.png 
Processing: Mitochondrial Rejuvenation or Exhaustion - Found 12 genes
Saved: DEG/Expression profiles/Mitochondria/GSE164425/Mitochondrial Rejuvenation or Exhaustion_expression_plot.png 
Processing: Integrins and Sestrin

In [12]:
generate_expression_plots(
  input_file = "Normalized/GSE239446.csv",
  output_dir = "DEG/Expression profiles/Mitochondria/GSE239446",
  condition_colors = c("iPSC" = "red", "NSC" = "green4", "Reprogrammed" = "blue"),
  condition_patterns = list("iPSC" = "iPSC", "NSC" = "NSC", "Reprogrammed" = "Reprogrammed")
)

Rows: 62707 Columns: 13
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr  (1): GENE_ID
dbl (12): GSM8422193_iPSC_GSE239446, GSM8422194_iPSC_GSE239446, GSM8422195_i...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


Condition assignments in data:

        iPSC          NSC Reprogrammed 
      188121       188121       376242 
Processing: Tracing Mitochondrial Markers - Found 17 genes
Saved: DEG/Expression profiles/Mitochondria/GSE239446/Tracing Mitochondrial Markers_expression_plot.png 
Processing: Alternative Splicing - Found 1 genes
Saved: DEG/Expression profiles/Mitochondria/GSE239446/Alternative Splicing_expression_plot.png 
Processing: Mitochondrial Dysfunction and Global Variability - Found 18 genes
Saved: DEG/Expression profiles/Mitochondria/GSE239446/Mitochondrial Dysfunction and Global Variability_expression_plot.png 
Processing: Bioenergetics from Gene Analysis - Found 18 genes
Saved: DEG/Expression profiles/Mitochondria/GSE239446/Bioenergetics from Gene Analysis_expression_plot.png 
Processing: Mitochondrial Rejuvenation or Exhaustion - Found 13 genes
Saved: DEG/Expression profiles/Mitochondria/GSE239446/Mitochondrial Rejuvenation or Exhaustion_expression_plot.png 
Processing: Integrins

In [14]:
generate_expression_plots(
  input_file = "Normalized/GSE252276.csv",
  output_dir = "DEG/Expression profiles/Mitochondria/GSE252276",
  condition_colors = c("AIPLwt" = "red", "AIPLco" = "green4", "control" = "blue"),
  condition_patterns = list("AIPLwt" = "AIPLwt", "AIPLco" = "AIPLco", "control" = "control")
)

Rows: 39376 Columns: 12
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr  (1): GENE_ID
dbl (11): RPE_GSE252276_AIPLwt_1_GSM7998492, RPE_GSE252276_AIPLwt_3_GSM79984...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


Condition assignments in data:

 AIPLco  AIPLwt control   Other 
 118128  118128  118128   78752 
Processing: Tracing Mitochondrial Markers - Found 17 genes
Saved: DEG/Expression profiles/Mitochondria/GSE252276/Tracing Mitochondrial Markers_expression_plot.png 
Processing: Alternative Splicing - Found 1 genes
Saved: DEG/Expression profiles/Mitochondria/GSE252276/Alternative Splicing_expression_plot.png 
Processing: Mitochondrial Dysfunction and Global Variability - Found 18 genes
Saved: DEG/Expression profiles/Mitochondria/GSE252276/Mitochondrial Dysfunction and Global Variability_expression_plot.png 
Processing: Bioenergetics from Gene Analysis - Found 18 genes
Saved: DEG/Expression profiles/Mitochondria/GSE252276/Bioenergetics from Gene Analysis_expression_plot.png 
Processing: Mitochondrial Rejuvenation or Exhaustion - Found 8 genes
Saved: DEG/Expression profiles/Mitochondria/GSE252276/Mitochondrial Rejuvenation or Exhaustion_expression_plot.png 
Processing: Integrins and Sestrins -

# DEG

In [46]:
run_DEG_analysis <- function(input_path, output_dir, samples, groups, logFC_cutoff = 0, comparison_label = "HSCinducediNKT_vs_NK") {
  if (!requireNamespace("BiocManager", quietly = TRUE)) install.packages("BiocManager")
  pkgs <- c("DESeq2", "ggplot2", "pheatmap", "EnhancedVolcano", "vsn")
  to_install <- pkgs[!pkgs %in% rownames(installed.packages())]
  if (length(to_install)) BiocManager::install(to_install, update = FALSE)
  invisible(lapply(pkgs, require, character.only = TRUE))
  
  if (!dir.exists(output_dir)) dir.create(output_dir, recursive = TRUE)
  
  counts_raw <- read.csv(input_path, header = TRUE, check.names = FALSE)
  rownames(counts_raw) <- make.unique(as.character(counts_raw[[1]]))
  counts <- counts_raw[, -1]
  counts <- as.matrix(round(counts))
  counts <- counts[, samples, drop = FALSE]
  
  condition <- factor(groups, levels = unique(groups))
  col_data <- data.frame(row.names = samples, condition = condition)
  
  dds <- DESeqDataSetFromMatrix(countData = counts, colData = col_data, design = ~ condition)
  keep <- rowSums(counts(dds)) >= 10
  dds <- dds[keep, ]
  
  dds <- DESeq(dds, minReplicatesForReplace = Inf)
  
  res <- results(dds,
                 contrast = c("condition", levels(condition)[2], levels(condition)[1]),
                 alpha = 0.05, cooksCutoff = FALSE)
  
  res_df <- as.data.frame(res)
  res_df$gene <- rownames(res_df)
  
  res_df$regulation <- "NS"
  if (logFC_cutoff == 0) {
    res_df$regulation[!is.na(res_df$padj) & res_df$padj < 0.05 & res_df$log2FoldChange > 0] <- "Up"
    res_df$regulation[!is.na(res_df$padj) & res_df$padj < 0.05 & res_df$log2FoldChange < 0] <- "Down"
  } else {
    res_df$regulation[!is.na(res_df$padj) & res_df$padj < 0.05 & res_df$log2FoldChange > logFC_cutoff] <- "Up"
    res_df$regulation[!is.na(res_df$padj) & res_df$padj < 0.05 & res_df$log2FoldChange < -logFC_cutoff] <- "Down"
  }
  
  write.csv(res_df[order(res_df$padj), ],
            file.path(output_dir, paste0("DEG_", comparison_label, "_logFC", logFC_cutoff, ".csv")),
            row.names = FALSE)
  
  sig_df <- res_df[res_df$regulation %in% c("Up", "Down"), ]
  write.csv(sig_df[order(sig_df$padj), ],
            file.path(output_dir, paste0("DEG_", comparison_label, "_logFC", logFC_cutoff, "_significant.csv")),
            row.names = FALSE)
  
  png(file.path(output_dir, paste0("Volcano_", comparison_label, "_logFC", logFC_cutoff, ".png")),
      width = 1200, height = 1000, res = 150)
  print(EnhancedVolcano(res_df,
                        lab = res_df$gene,
                        x = "log2FoldChange",
                        y = "padj",
                        pCutoff = 0.05,
                        FCcutoff = logFC_cutoff,
                        title = paste(comparison_label, " (logFC", logFC_cutoff, ")", sep = ""),
                        legendPosition = "right"))
  dev.off()
  
  if (nrow(sig_df) >= 2) {
    vst_dds <- vst(dds, blind = FALSE)
    top_genes <- rownames(sig_df[order(sig_df$padj), ])[1:min(300, nrow(sig_df))]
    cat("Number of top DEGs for heatmap:", length(top_genes), "\n")
    if (length(top_genes) > 0) {
      mat <- assay(vst_dds)[top_genes, samples, drop = FALSE]
      # Check matrix valid
      if (!is.null(mat) && nrow(mat) > 0 && ncol(mat) > 0) {
        if (any(is.na(mat)) || any(is.infinite(mat))) {
          stop("Matrix contains NA or infinite values, cannot plot heatmap.")
        }
        mat <- mat - rowMeans(mat)
        ann <- data.frame(condition = condition)
        rownames(ann) <- samples
        
        # Close any open devices before plotting
        while(dev.cur() > 1) dev.off()
        
        heatmap_file <- file.path(output_dir, paste0("Heatmap_top300_", comparison_label, "_logFC", logFC_cutoff, ".png"))
        png(heatmap_file, width = 1400, height = 1400, res = 150)
        pheatmap(mat,
                 annotation_col = ann,
                 fontsize_row = 6,
                 show_rownames = FALSE,
                 main = paste("Top 300 DEGs:", comparison_label, "(logFC", logFC_cutoff, ")"))
        dev.off()
        
        # Verify file creation and size
        if (!file.exists(heatmap_file) || file.info(heatmap_file)$size == 0) {
          stop("Heatmap file was not created or is empty.")
        } else {
          cat("Heatmap successfully created:", heatmap_file, "\n")
        }
      } else {
        cat("Matrix for heatmap is empty, skipping heatmap generation.\n")
      }
    } else {
      cat("No top DEGs available for heatmap.\n")
    }
  } else {
    cat("Insufficient DEGs for heatmap generation.\n")
  }
  
  cat(paste0("✓ DEG analysis complete for ", comparison_label, " (logFC", logFC_cutoff, ")\n"))
}


In [47]:
samples <- c(
  "NK_GSE164425_control_NK_1",
  "NK_GSE164425_control_NK_2",
  "HSCinducedNK_GSE164425_HSCinducediNKT_CB_1",
  "HSCinducedNK_GSE164425_HSCinducediNKT_CB_2",
  "HSCinducedNK_GSE164425_HSCinducediNKT_CB_3",
  "HSCinducedNK_GSE164425_HSCinducediNKT_PBSC_1",
  "HSCinducedNK_GSE164425_HSCinducediNKT_PBSC_3",
  "HSCinducedNK_GSE164425_HSCinducediNKT_PBSC_4"
)
groups <- c("NK", "NK", "HSCinducediNKT", "HSCinducediNKT", "HSCinducediNKT",
            "HSCinducediNKT", "HSCinducediNKT", "HSCinducediNKT")

run_DEG_analysis("Raw/New/GSE164425_label.csv", "DEG/Results/GSE164425_logFC0", samples, groups, 0, "HSCinducediNKT_vs_NK")
run_DEG_analysis("Raw/New/GSE164425_label.csv", "DEG/Results/GSE164425_logFC1", samples, groups, 1, "HSCinducediNKT_vs_NK")


converting counts to integer mode

estimating size factors

estimating dispersions

gene-wise dispersion estimates

mean-dispersion relationship

final dispersion estimates

fitting model and testing



Number of top DEGs for heatmap: 300 
Heatmap successfully created: DEG/Results/GSE164425_logFC0/Heatmap_top300_HSCinducediNKT_vs_NK_logFC0.png 
✓ DEG analysis complete for HSCinducediNKT_vs_NK (logFC0)


converting counts to integer mode

estimating size factors

estimating dispersions

gene-wise dispersion estimates

mean-dispersion relationship

final dispersion estimates

fitting model and testing



Number of top DEGs for heatmap: 300 
Heatmap successfully created: DEG/Results/GSE164425_logFC1/Heatmap_top300_HSCinducediNKT_vs_NK_logFC1.png 
✓ DEG analysis complete for HSCinducediNKT_vs_NK (logFC1)


In [48]:
samples1 <- c(
  "RPE_GSE252276_AIPLwt_1_GSM7998492",
  "RPE_GSE252276_AIPLwt_2_GSM7998499",
  "RPE_GSE252276_AIPLwt_3_GSM7998493",
  "RPE_GSE252276_control_1_GSM7998496",
  "RPE_GSE252276_control_2_GSM7998501",
  "RPE_GSE252276_control_3_GSM7998502"
)
groups1 <- c(
  "AIPLwt", "AIPLwt", "AIPLwt",
  "control", "control", "control"
)

run_DEG_analysis("Raw/New/GSE252276_label.csv", "DEG/Results/GSE252276_AIPLwt_vs_control_logFC0", samples1, groups1, 0, "AIPLwt_vs_control")
run_DEG_analysis("Raw/New/GSE252276_label.csv", "DEG/Results/GSE252276_AIPLwt_vs_control_logFC1", samples1, groups1, 1, "AIPLwt_vs_control")


converting counts to integer mode

  it appears that the last variable in the design formula, 'condition',
  has a factor level, 'control', which is not the reference level. we recommend
  to use factor(...,levels=...) or relevel() to set this as the reference level
  before proceeding. for more information, please see the 'Note on factor levels'
  in vignette('DESeq2').

estimating size factors

estimating dispersions

gene-wise dispersion estimates

mean-dispersion relationship

final dispersion estimates

fitting model and testing



Number of top DEGs for heatmap: 300 
Heatmap successfully created: DEG/Results/GSE252276_AIPLwt_vs_control_logFC0/Heatmap_top300_AIPLwt_vs_control_logFC0.png 
✓ DEG analysis complete for AIPLwt_vs_control (logFC0)


converting counts to integer mode

  it appears that the last variable in the design formula, 'condition',
  has a factor level, 'control', which is not the reference level. we recommend
  to use factor(...,levels=...) or relevel() to set this as the reference level
  before proceeding. for more information, please see the 'Note on factor levels'
  in vignette('DESeq2').

estimating size factors

estimating dispersions

gene-wise dispersion estimates

mean-dispersion relationship

final dispersion estimates

fitting model and testing



Number of top DEGs for heatmap: 300 
Heatmap successfully created: DEG/Results/GSE252276_AIPLwt_vs_control_logFC1/Heatmap_top300_AIPLwt_vs_control_logFC1.png 
✓ DEG analysis complete for AIPLwt_vs_control (logFC1)


In [49]:
samples2 <- c(
  "RPE_GSE252276_AIPLco_1_GSM7998500",
  "RPE_GSE252276_AIPLco_2_GSM7998494",
  "RPE_GSE252276_AIPLco_3_GSM7998495",
  "RPE_GSE252276_control_1_GSM7998496",
  "RPE_GSE252276_control_2_GSM7998501",
  "RPE_GSE252276_control_3_GSM7998502"
)
groups2 <- c(
  "AIPLco", "AIPLco", "AIPLco",
  "control", "control", "control"
)

run_DEG_analysis("Raw/New/GSE252276_label.csv", "DEG/Results/GSE252276_AIPLco_vs_control_logFC0", samples2, groups2, 0, "AIPLco_vs_control")
run_DEG_analysis("Raw/New/GSE252276_label.csv", "DEG/Results/GSE252276_AIPLco_vs_control_logFC1", samples2, groups2, 1, "AIPLco_vs_control")


converting counts to integer mode

  it appears that the last variable in the design formula, 'condition',
  has a factor level, 'control', which is not the reference level. we recommend
  to use factor(...,levels=...) or relevel() to set this as the reference level
  before proceeding. for more information, please see the 'Note on factor levels'
  in vignette('DESeq2').

estimating size factors

estimating dispersions

gene-wise dispersion estimates

mean-dispersion relationship

final dispersion estimates

fitting model and testing



Number of top DEGs for heatmap: 300 
Heatmap successfully created: DEG/Results/GSE252276_AIPLco_vs_control_logFC0/Heatmap_top300_AIPLco_vs_control_logFC0.png 
✓ DEG analysis complete for AIPLco_vs_control (logFC0)


converting counts to integer mode

  it appears that the last variable in the design formula, 'condition',
  has a factor level, 'control', which is not the reference level. we recommend
  to use factor(...,levels=...) or relevel() to set this as the reference level
  before proceeding. for more information, please see the 'Note on factor levels'
  in vignette('DESeq2').

estimating size factors

estimating dispersions

gene-wise dispersion estimates

mean-dispersion relationship

final dispersion estimates

fitting model and testing



Number of top DEGs for heatmap: 300 
Heatmap successfully created: DEG/Results/GSE252276_AIPLco_vs_control_logFC1/Heatmap_top300_AIPLco_vs_control_logFC1.png 
✓ DEG analysis complete for AIPLco_vs_control (logFC1)


In [50]:
samples3 <- c(
  "RPE_GSE252276_AIPLco_1_GSM7998500",
  "RPE_GSE252276_AIPLco_2_GSM7998494",
  "RPE_GSE252276_AIPLco_3_GSM7998495",
  "RPE_GSE252276_AIPLwt_1_GSM7998492",
  "RPE_GSE252276_AIPLwt_2_GSM7998499",
  "RPE_GSE252276_AIPLwt_3_GSM7998493"
)
groups3 <- c(
  "AIPLco", "AIPLco", "AIPLco",
  "AIPLwt", "AIPLwt", "AIPLwt"
)

run_DEG_analysis("Raw/New/GSE252276_label.csv", "DEG/Results/GSE252276_AIPLco_vs_AIPLwt_logFC0", samples3, groups3, 0, "AIPLco_vs_AIPLwt")
run_DEG_analysis("Raw/New/GSE252276_label.csv", "DEG/Results/GSE252276_AIPLco_vs_AIPLwt_logFC1", samples3, groups3, 1, "AIPLco_vs_AIPLwt")


converting counts to integer mode

estimating size factors

estimating dispersions

gene-wise dispersion estimates

mean-dispersion relationship

final dispersion estimates

fitting model and testing



Number of top DEGs for heatmap: 300 
Heatmap successfully created: DEG/Results/GSE252276_AIPLco_vs_AIPLwt_logFC0/Heatmap_top300_AIPLco_vs_AIPLwt_logFC0.png 
✓ DEG analysis complete for AIPLco_vs_AIPLwt (logFC0)


converting counts to integer mode

estimating size factors

estimating dispersions

gene-wise dispersion estimates

mean-dispersion relationship

final dispersion estimates

fitting model and testing



Number of top DEGs for heatmap: 300 
Heatmap successfully created: DEG/Results/GSE252276_AIPLco_vs_AIPLwt_logFC1/Heatmap_top300_AIPLco_vs_AIPLwt_logFC1.png 
✓ DEG analysis complete for AIPLco_vs_AIPLwt (logFC1)


In [51]:
samples1 <- c(
  "reprogrammed_GSE139273_iSN_1_GSM4135904",
  "reprogrammed_GSE139273_iSN_2_GSM4135905",
  "reprogrammed_GSE139273_iSN_3_GSM4135906",
  "neuron_GSE139273_control_hDRG_1_GSM4135900"
)
groups1 <- c(
  "iSN", "iSN", "iSN",
  "hDRG"
)

run_DEG_analysis("Raw/New/GSE139273_label.csv", "DEG/Results/GSE139273_iSN_vs_hDRG_logFC0", samples1, groups1, 0, "iSN_vs_hDRG")
run_DEG_analysis("Raw/New/GSE139273_label.csv", "DEG/Results/GSE139273_iSN_vs_hDRG_logFC1", samples1, groups1, 1, "iSN_vs_hDRG")


converting counts to integer mode

estimating size factors

estimating dispersions

gene-wise dispersion estimates

mean-dispersion relationship

final dispersion estimates

fitting model and testing



Number of top DEGs for heatmap: 300 
Heatmap successfully created: DEG/Results/GSE139273_iSN_vs_hDRG_logFC0/Heatmap_top300_iSN_vs_hDRG_logFC0.png 
✓ DEG analysis complete for iSN_vs_hDRG (logFC0)


converting counts to integer mode

estimating size factors

estimating dispersions

gene-wise dispersion estimates

mean-dispersion relationship

final dispersion estimates

fitting model and testing



Number of top DEGs for heatmap: 300 
Heatmap successfully created: DEG/Results/GSE139273_iSN_vs_hDRG_logFC1/Heatmap_top300_iSN_vs_hDRG_logFC1.png 
✓ DEG analysis complete for iSN_vs_hDRG (logFC1)


In [52]:
samples2 <- c(
  "iPSC_GSE139273_iPSC_1_GSM4135901",
  "iPSC_GSE139273_iPSC_2_GSM4135902",
  "iPSC_GSE139273_iPSC_3_GSM4135903",
  "reprogrammed_GSE139273_iSN_1_GSM4135904",
  "reprogrammed_GSE139273_iSN_2_GSM4135905",
  "reprogrammed_GSE139273_iSN_3_GSM4135906"
)
groups2 <- c(
  "iPSC", "iPSC", "iPSC",
  "iSN", "iSN", "iSN"
)

run_DEG_analysis("Raw/New/GSE139273_label.csv", "DEG/Results/GSE139273_iPSC_vs_iSN_logFC0", samples2, groups2, 0, "iPSC_vs_iSN")
run_DEG_analysis("Raw/New/GSE139273_label.csv", "DEG/Results/GSE139273_iPSC_vs_iSN_logFC1", samples2, groups2, 1, "iPSC_vs_iSN")


converting counts to integer mode

estimating size factors

estimating dispersions

gene-wise dispersion estimates

mean-dispersion relationship

final dispersion estimates

fitting model and testing

Warning message:
"One or more p-values is 0. Converting to 10^-1 * current lowest non-zero p-value..."


Number of top DEGs for heatmap: 300 
Heatmap successfully created: DEG/Results/GSE139273_iPSC_vs_iSN_logFC0/Heatmap_top300_iPSC_vs_iSN_logFC0.png 
✓ DEG analysis complete for iPSC_vs_iSN (logFC0)


converting counts to integer mode

estimating size factors

estimating dispersions

gene-wise dispersion estimates

mean-dispersion relationship

final dispersion estimates

fitting model and testing

Warning message:
"One or more p-values is 0. Converting to 10^-1 * current lowest non-zero p-value..."


Number of top DEGs for heatmap: 300 
Heatmap successfully created: DEG/Results/GSE139273_iPSC_vs_iSN_logFC1/Heatmap_top300_iPSC_vs_iSN_logFC1.png 
✓ DEG analysis complete for iPSC_vs_iSN (logFC1)


In [53]:
samples3 <- c(
  "iPSC_GSE139273_iPSC_1_GSM4135901",
  "iPSC_GSE139273_iPSC_2_GSM4135902",
  "iPSC_GSE139273_iPSC_3_GSM4135903",
  "neuron_GSE139273_control_hDRG_1_GSM4135900"
)
groups3 <- c(
  "iPSC", "iPSC", "iPSC",
  "hDRG"
)

run_DEG_analysis("Raw/New/GSE139273_label.csv", "DEG/Results/GSE139273_iPSC_vs_hDRG_logFC0", samples3, groups3, 0, "iPSC_vs_hDRG")
run_DEG_analysis("Raw/New/GSE139273_label.csv", "DEG/Results/GSE139273_iPSC_vs_hDRG_logFC1", samples3, groups3, 1, "iPSC_vs_hDRG")


converting counts to integer mode

estimating size factors

estimating dispersions

gene-wise dispersion estimates

mean-dispersion relationship

final dispersion estimates

fitting model and testing

Warning message:
"One or more p-values is 0. Converting to 10^-1 * current lowest non-zero p-value..."


Number of top DEGs for heatmap: 300 
Heatmap successfully created: DEG/Results/GSE139273_iPSC_vs_hDRG_logFC0/Heatmap_top300_iPSC_vs_hDRG_logFC0.png 
✓ DEG analysis complete for iPSC_vs_hDRG (logFC0)


converting counts to integer mode

estimating size factors

estimating dispersions

gene-wise dispersion estimates

mean-dispersion relationship

final dispersion estimates

fitting model and testing

Warning message:
"One or more p-values is 0. Converting to 10^-1 * current lowest non-zero p-value..."


Number of top DEGs for heatmap: 300 
Heatmap successfully created: DEG/Results/GSE139273_iPSC_vs_hDRG_logFC1/Heatmap_top300_iPSC_vs_hDRG_logFC1.png 
✓ DEG analysis complete for iPSC_vs_hDRG (logFC1)


In [54]:
samples <- c(
  "RNA_14dOb_BM_rep1", "RNA_7dOb_BM_rep1", "RNA_3dOb_BM_rep1", "RNA_1dOb_BM_rep1", "RNA_4hOb_BM_rep1", "RNA_Msc_BM_rep1",
  "RNA_14dOb_BM_rep2", "RNA_7dOb_BM_rep2", "RNA_3dOb_BM_rep2", "RNA_1dOb_BM_rep2", "RNA_4hOb_BM_rep2", "RNA_Msc_BM_rep2",
  "RNA_14dOb_BM_rep3", "RNA_7dOb_BM_rep3", "RNA_3dOb_BM_rep3", "RNA_1dOb_BM_rep3", "RNA_4hOb_BM_rep3", "RNA_Msc_BM_rep3"
)

groups <- c(
  "BM", "BM", "BM", "BM", "BM", "Undifferentiated",
  "BM", "BM", "BM", "BM", "BM", "Undifferentiated",
  "BM", "BM", "BM", "BM", "BM", "Undifferentiated"
)

run_DEG_analysis("Raw/New/GSE113253.csv", "DEG/Results/GSE113253_BM_vs_Msc_logFC0", samples, groups, 0, "BM_vs_Undifferentiated")
run_DEG_analysis("Raw/New/GSE113253.csv", "DEG/Results/GSE113253_BM_vs_Msc_logFC1", samples, groups, 1, "BM_vs_Undifferentiated")

converting counts to integer mode

estimating size factors

estimating dispersions

gene-wise dispersion estimates

mean-dispersion relationship

final dispersion estimates

fitting model and testing



Number of top DEGs for heatmap: 300 
Heatmap successfully created: DEG/Results/GSE113253_BM_vs_Msc_logFC0/Heatmap_top300_BM_vs_Undifferentiated_logFC0.png 
✓ DEG analysis complete for BM_vs_Undifferentiated (logFC0)


converting counts to integer mode

estimating size factors

estimating dispersions

gene-wise dispersion estimates

mean-dispersion relationship

final dispersion estimates

fitting model and testing



Number of top DEGs for heatmap: 271 
Heatmap successfully created: DEG/Results/GSE113253_BM_vs_Msc_logFC1/Heatmap_top300_BM_vs_Undifferentiated_logFC1.png 
✓ DEG analysis complete for BM_vs_Undifferentiated (logFC1)


In [55]:
samples <- c(
  "RNA_14dOb_AT_rep1", "RNA_7dOb_AT_rep1", "RNA_3dOb_AT_rep1", "RNA_1dOb_AT_rep1", "RNA_4hOb_AT_rep1", "RNA_Msc_AT_rep1",
  "RNA_14dOb_AT_rep2", "RNA_7dOb_AT_rep2", "RNA_3dOb_AT_rep2", "RNA_1dOb_AT_rep2", "RNA_4hOb_AT_rep2", "RNA_Msc_AT_rep2"
)

groups <- c(
  "AT", "AT", "AT", "AT", "AT", "Undifferentiated",
  "AT", "AT", "AT", "AT", "AT", "Undifferentiated"
)

run_DEG_analysis("Raw/New/GSE113253.csv", "DEG/Results/GSE113253_AT_vs_Msc_logFC0", samples, groups, 0, "AT_vs_Undifferentiated")
run_DEG_analysis("Raw/New/GSE113253.csv", "DEG/Results/GSE113253_AT_vs_Msc_logFC1", samples, groups, 1, "AT_vs_Undifferentiated")

converting counts to integer mode

estimating size factors

estimating dispersions

gene-wise dispersion estimates

mean-dispersion relationship

final dispersion estimates

fitting model and testing



Number of top DEGs for heatmap: 300 
Heatmap successfully created: DEG/Results/GSE113253_AT_vs_Msc_logFC0/Heatmap_top300_AT_vs_Undifferentiated_logFC0.png 
✓ DEG analysis complete for AT_vs_Undifferentiated (logFC0)


converting counts to integer mode

estimating size factors

estimating dispersions

gene-wise dispersion estimates

mean-dispersion relationship

final dispersion estimates

fitting model and testing



Number of top DEGs for heatmap: 217 
Heatmap successfully created: DEG/Results/GSE113253_AT_vs_Msc_logFC1/Heatmap_top300_AT_vs_Undifferentiated_logFC1.png 
✓ DEG analysis complete for AT_vs_Undifferentiated (logFC1)
